In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
datasets = {
    "county": "county_name.csv",
    "population": "population_2025.csv",
    "housing": "census_income_housing.csv",
    "energy": "energy_consumption_by_county.csv",
    "data_centers": "data_centers_by_county.csv",
    "flood": "flood_by_counties.csv",
    "lake": "lakes_by_county.csv",
    "military": "military_by_county.csv",
    "nuclear_plant": "nuclear_plants_cleaned.csv",
    "seismic": "seismic_hazard.csv",
    "transmission_lines": "transmission_lines_by_county.csv",
    "wetlands": "wetlands_cleaned.csv",
    "rivers": "rivers_by_county.csv"
}

processed_dir = Path.cwd().parent / "processed_data"
processed_dir

WindowsPath('c:/Users/brian/OneDrive - University of Illinois - Urbana/team-nuclear-family/processed_data')

In [3]:
df = {}

for data, dataset in datasets.items():
    df[data] = pd.read_csv(processed_dir/dataset)
    print(f"done {data}")

done county
done population
done housing
done energy
done data_centers
done flood
done lake
done military
done nuclear_plant
done seismic
done transmission_lines
done wetlands
done rivers


In [4]:
print("Columns of all datasets before dropping: \n")
for data, dataset in df.items():
    if "Unnamed: 0" in dataset.columns:
        df[data] = dataset.drop(columns = "Unnamed: 0")
    print(f"{data} columns: {df[data].columns}")

Columns of all datasets before dropping: 

county columns: Index(['GEOID', 'NAMELSAD'], dtype='object')
population columns: Index(['state_name', 'county_name', 'state_fips', 'county_fips', 'geoid',
       'state_abbr', 'population_2025'],
      dtype='object')
housing columns: Index(['county_name', 'median_household_income', 'housing_units', 'state_fips',
       'county_fips', 'geoid', 'state_name', 'state_abbr'],
      dtype='object')
energy columns: Index(['geo_id', 'county_name', 'total_energy_consumption_mwh', 'Unnamed: 3',
       'Unnamed: 4', 'Unnamed: 5'],
      dtype='object')
data_centers columns: Index(['county_name', 'geoid', 'data_centers_count'], dtype='object')
flood columns: Index(['GEOID', 'sfha_area', 'county_name', 'county_area', 'pct_sfha',
       'sfha_area_log'],
      dtype='object')
lake columns: Index(['geoid', 'county_name', 'lake_count', 'total_lake_area', 'avg_vol',
       'avg_depth', 'avg_discharge', 'dist_to_lakes_km'],
      dtype='object')
military colum

In [5]:
df["county"].columns = ["geo_id", "county_name"]

# population
df["population"] = df["population"].drop(columns = ["state_name", "county_name", "state_fips", "county_fips", "state_abbr"])
df["population"].columns = ["geo_id", "population"]

# housing
df["housing"] = df["housing"].drop(columns = ["county_name", "state_fips", "county_fips", "state_name", "state_abbr"])
df["housing"].columns = ["median_household_income", "housing_units", "geo_id"]

# energy
df["energy"] = df["energy"].drop(columns = ["county_name", 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5'])
df["energy"]["total_energy_consumption_mwh"] = df["energy"]["total_energy_consumption_mwh"].str.replace(",", "").astype(float)

# data centers
df["data_centers"] = df["data_centers"].rename(columns = {"geoid": "geo_id"}).drop(columns = "county_name")

# flood
df["flood"] = df["flood"].rename(columns = {"GEOID": "geo_id"}).drop(columns = ["county_name", "county_area", "sfha_area_log"])

# lake
df["lake"] = df["lake"].rename(columns = {"geoid": "geo_id"}).drop(columns = "county_name")
df["lake"]['avg_discharge'] = df["lake"]['avg_discharge'].mask(df['lake']['avg_discharge'] < 0, np.nan)
df['lake']['avg_discharge'] = df['lake']['avg_discharge'].fillna(0)

# military 
df["military"] = df["military"].drop(columns = "county_name")

# nuclear plants
df["nuclear_plant"] = df["nuclear_plant"].drop(columns = ["county_name", "plant_name"])

# seismic
df["seismic"] = df["seismic"].drop(columns = ["state_fips", "county_fips", "county_name", 
                                              "state_abbr", "lat", "lon", 'ss_spectral_02s', 
                                              's1_spectral_1s', 'sds_design', 'sd1_design'])
df["seismic"].columns = ["geo_id", "pga_max"]

# transmission lines
df["transmission_lines"] = df["transmission_lines"].drop(columns = "county_name")

# wetlands 
df["wetlands"] = df["wetlands"].drop(columns = ["county_name", "named_wetlands"])

# rivers
df["rivers"] = df["rivers"].drop(columns = "county_name")

print("Columns of all datasets after dropping: \n")

for data, dataset in df.items():
    print(f"{data} columns: {df[data].columns}")

Columns of all datasets after dropping: 

county columns: Index(['geo_id', 'county_name'], dtype='object')
population columns: Index(['geo_id', 'population'], dtype='object')
housing columns: Index(['median_household_income', 'housing_units', 'geo_id'], dtype='object')
energy columns: Index(['geo_id', 'total_energy_consumption_mwh'], dtype='object')
data_centers columns: Index(['geo_id', 'data_centers_count'], dtype='object')
flood columns: Index(['geo_id', 'sfha_area', 'pct_sfha'], dtype='object')
lake columns: Index(['geo_id', 'lake_count', 'total_lake_area', 'avg_vol', 'avg_depth',
       'avg_discharge', 'dist_to_lakes_km'],
      dtype='object')
military columns: Index(['geo_id', 'military_count', 'total_military_area_m', 'pct_military'], dtype='object')
nuclear_plant columns: Index(['geo_id', 'plant_count'], dtype='object')
seismic columns: Index(['geo_id', 'pga_max'], dtype='object')
transmission_lines columns: Index(['geo_id', 'distance_to_lines_km', 'transmission_lines_count',

In [6]:
# some dataset might remove the 0 at the beginning of the geoid of some counties

for name in df:
    if "geo_id" in df[name].columns:
        df[name]["geo_id"] = df[name]["geo_id"].astype(str).str.zfill(5)

In [7]:
df_master = df["county"].merge(df["population"], on = "geo_id", how = "left") \
                        .merge(df["housing"], on = "geo_id", how = "left") \
                        .merge(df["energy"], on = "geo_id", how = "left") \
                        .merge(df["data_centers"], on = "geo_id", how = "left") \
                        .merge(df["flood"], on = "geo_id", how = "left") \
                        .merge(df["lake"], on = "geo_id", how = "left") \
                        .merge(df["wetlands"], on = "geo_id", how = "left") \
                        .merge(df["rivers"], on = "geo_id", how = "left") \
                        .merge(df["military"], on = "geo_id", how = "left") \
                        .merge(df["nuclear_plant"], on = "geo_id", how = "left") \
                        .merge(df["seismic"], on = "geo_id", how = "left") \
                        .merge(df["transmission_lines"], on = "geo_id", how = "left")
df_master

,geo_id,county_name,population,median_household_income,housing_units,total_energy_consumption_mwh,data_centers_count,sfha_area,pct_sfha,lake_count,...,total_rivers_mile,military_count,total_military_area_m,pct_military,plant_count,pga_max,distance_to_lines_km,transmission_lines_count,max_voltage,average_voltage
0,40075,Kiowa County,"8,181",42679.0,4700.0,157935.0,NaN,2.112674e+08,0.079146,10,...,688.07,NaN,NaN,NaN,NaN,0.180,16.455467,10.0,138.0,110.400000
1,46079,Lake County,"10,993",74884.0,5714.0,170505.0,NaN,1.890356e+08,0.126898,48,...,47.64,NaN,NaN,NaN,NaN,0.051,2.165857,18.0,69.0,69.000000
2,37033,Caswell County,"22,563",56999.0,10493.0,295896.0,NaN,6.618573e+07,0.059609,4,...,178.23,NaN,NaN,NaN,NaN,0.082,10.562934,3.0,230.0,230.000000
3,48377,Presidio County,"5,433",29012.0,3396.0,106309.0,NaN,NaN,NaN,0,...,726.40,NaN,NaN,NaN,NaN,0.150,6.887895,4.0,69.0,69.000000
4,39057,Greene County,"174,322",81243.0,71471.0,2239244.0,NaN,1.002618e+08,0.092999,8,...,276.85,1.0,2.104756e+07,0.019523,NaN,0.097,1.588568,44.0,345.0,112.909091
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3230,53065,Stevens County,"49,668",62381.0,22312.0,536848.0,NaN,1.749183e+05,0.000027,27,...,798.02,NaN,NaN,NaN,NaN,0.150,71.466142,0.0,0.0,0.000000
3231,19177,Van Buren County,"7,131",58417.0,3509.0,126206.0,NaN,1.457009e+08,0.114681,5,...,321.91,NaN,NaN,NaN,NaN,0.065,6.011057,8.0,69.0,69.000000
3232,31073,Gosper County,"1,803",76583.0,1193.0,33218.0,NaN,8.760107e+07,0.073095,4,...,97.63,NaN,NaN,NaN,NaN,0.043,2.795926,19.0,115.0,93.210526
3233,28095,Monroe County,"33,318",51190.0,16739.0,582913.0,NaN,4.869952e+08,0.243543,12,...,319.31,NaN,NaN,NaN,NaN,0.180,7.967100,17.0,161.0,139.352941


In [8]:
df_master.isnull().sum()

geo_id                             0
county_name                        0
population                        91
median_household_income           14
housing_units                     13
total_energy_consumption_mwh      22
data_centers_count              2991
sfha_area                         75
pct_sfha                          75
lake_count                         0
total_lake_area                    0
avg_vol                            0
avg_depth                          0
avg_discharge                      0
dist_to_lakes_km                   0
wetland_count                   1672
distance_to_rivers_km              0
rivers_count                      60
total_rivers_mile                 60
military_count                  2707
total_military_area_m           2707
pct_military                    2707
plant_count                     3181
pga_max                           15
distance_to_lines_km               0
transmission_lines_count           0
max_voltage                        0
a

In [9]:
df_master.columns

Index(['geo_id', 'county_name', 'population', 'median_household_income',
       'housing_units', 'total_energy_consumption_mwh', 'data_centers_count',
       'sfha_area', 'pct_sfha', 'lake_count', 'total_lake_area', 'avg_vol',
       'avg_depth', 'avg_discharge', 'dist_to_lakes_km', 'wetland_count',
       'distance_to_rivers_km', 'rivers_count', 'total_rivers_mile',
       'military_count', 'total_military_area_m', 'pct_military',
       'plant_count', 'pga_max', 'distance_to_lines_km',
       'transmission_lines_count', 'max_voltage', 'average_voltage'],
      dtype='object')

In [10]:
# fill null values with 0 for certain columns

fill_zero = [
    "data_centers_count",      
    "lake_count",      
    "total_lake_area",  
    "avg_vol",  
    "avg_depth",  
    "avg_discharge",
    "wetland_count", 
    "rivers_count", 
    "total_rivers_mile",
    "military_count",
    "total_military_area_m",
    "pct_military",
    "plant_count", 
    "transmission_lines_count",
    "pga_max",
    'sfha_area', 
    'pct_sfha'
]

df_master[fill_zero] = df_master[fill_zero].fillna(0)
df_master.isnull().sum()

geo_id                           0
county_name                      0
population                      91
median_household_income         14
housing_units                   13
total_energy_consumption_mwh    22
data_centers_count               0
sfha_area                        0
pct_sfha                         0
lake_count                       0
total_lake_area                  0
avg_vol                          0
avg_depth                        0
avg_discharge                    0
dist_to_lakes_km                 0
wetland_count                    0
distance_to_rivers_km            0
rivers_count                     0
total_rivers_mile                0
military_count                   0
total_military_area_m            0
pct_military                     0
plant_count                      0
pga_max                          0
distance_to_lines_km             0
transmission_lines_count         0
max_voltage                      0
average_voltage                  0
dtype: int64

**Notes for including `sfha_area` and `pct_sfha` (indicators of flood risks) to fill null values with 0:**
- The original flood database from FEMA includes 50 different datasets of **flood regions** in 50 states (not based on counties). To standardize the dataset to county unit, we used spatial overlay in `geopandas` and the county boundaries dataset (`../raw_data/county_boundaries_2025`) to find the overlapping area between a flood region and a county.

- When a county does not have a flood region, it is not included in the joined dataset, meaning that its severe flood hazard area (SFHA) is 0. 

In [11]:
df_master.to_csv("../processed_data/final_dataset.csv")

**Variables in final_dataset.csv**

- `geo_id`, `county_name`: unique ID of counties and names (names might be duplicated) 

- `population`: population of county in 2025 (released in March 2026 by Census)

- `median_household_income`: 

- `housing_units`: 

- `total_energy_consumption_mwh`: total energy consumption of the county in 2024 in MWh	

- `data_centers_count`: number of data centers in the county 

- `sfha_area`: severe flood hazard area (SFHA) in the county

- `pct_sfha`: proportion of the county that is SFHA

- `lake_count`: number of lakes that intersectsthe county

- `total_lake_area`: total lakes area in the county

- `avg_vol`: average volume of lakes in the county 

- `avg_depth`: average depth of lakes in the county 

- `avg_discharge`: average discharge of lakes in the county (discharge = the volume of water flowing out of it over a specific period)

- `dist_to_lakes_km`: distance from the county’s centroid to the nearest lake in kilometers

- `wetland_count`: number of wetlands that intersect the county

- `distance_to_rivers_km`: distance from the county’s centroid to the nearest river in kilometers

- `rivers_count`: number of rivers that intersect the county

- `total_rivers_mile`: total miles of rivers in the county

- `military_count`: number of military installations that intersects in the county

- `total_military_area_m`: total area of military installations in square meters

- `pct_military`: proportion of the county that are military installations 

- `plant_count`: number of nuclear plants in the county

- `pga_max`: peak ground acceleration (measure seismic risk)

- `distance_to_lines_km`: distance from the county’s centroid to the nearest transmission line in kilometers

- `transmission_lines_count`: number of transmission lines in the county

- `max_voltage`: maximum voltage of a transmission line in the county in kV

- `average_voltage`: average voltage of a transmission line in the county in kV